# Purpose:
- Compare data in lims and codeocean
    - I can't confirm if the asset is good. (cannot filter bad assets): can only do this in codeocean.
    - Instead, I can take a look at raw data (usually good) and see if there is anything absent in codeocean
- make csv file for uploading and triggering
- Follow up of 250418_gcamp8_data_curation_lims.ipynb
- Note: 
    - One error session (no video file in lims): 1366195973 (719363_2024-05-13)
## Use codeocean env

In [3]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

import codeocean
from codeocean.components import SearchFilter, SearchFilterRange
from codeocean.data_asset import DataAssetSearchParams

import aind_session
from lamf_analysis.code_ocean import capsule_data_utils as cdu

client = codeocean.CodeOcean(domain=os.getenv("CODEOCEAN_DOMAIN"), 
                             token=os.getenv("CODEOCEAN_TOKEN"))

%load_ext autoreload
%autoreload 2

In [18]:
data_asset = client.data_assets.get_data_asset(data_asset_id="bfbe8ed8-9ce8-4d35-a88e-8d887b3e13cc")


In [21]:
data_asset.files

176

In [20]:
dir(data_asset)

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'app_parameters',
 'contained_data_assets',
 'created',
 'custom_metadata',
 'description',
 'failure_reason',
 'files',
 'from_dict',
 'from_json',
 'id',
 'last_transferred',
 'last_used',
 'mount',
 'name',
 'provenance',
 'schema',
 'size',
 'source_bucket',
 'state',
 'tags',
 'to_dict',
 'to_json',
 'transfer_error',
 'type']

# Load lims curation

In [42]:
load_dir = Path(r'\\allen\programs\mindscope\workgroups\learning\pilots\GCaMP8')
plane_save_fn = load_dir / 'gcamp8_plane_info_250418.csv'
gcamp_info = pd.read_csv(plane_save_fn)

session_save_fn = load_dir / 'gcamp8_session_info_250418.csv'
gcamp_session_info = pd.read_csv(session_save_fn)

In [43]:
gcamp_session_info

,ophys_session_id,gcamp,full_genotype,mouse_id,date_of_acquisition,session_type,session_storage_directory,project
0,1410496548,oi4_homo,Oi4(TIT2L-jGCaMP8s-RiboL1-WPRE-ICL-IRES-tTA2-W...,759731,2024-12-16_19-32-04,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment
1,1409915431,oi4_homo,Oi4(TIT2L-jGCaMP8s-RiboL1-WPRE-ICL-IRES-tTA2-W...,759730,2024-12-12_16-46-12,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment
2,1410921088,oi4_homo,Oi4(TIT2L-jGCaMP8s-RiboL1-WPRE-ICL-IRES-tTA2-W...,759730,2024-12-19_17-07-01,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment
3,1409540466,oi4_homo,Oi4(TIT2L-jGCaMP8s-RiboL1-WPRE-ICL-IRES-tTA2-W...,759731,2024-12-10_17-14-09,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment
4,1409931438,oi4_homo,Oi4(TIT2L-jGCaMP8s-RiboL1-WPRE-ICL-IRES-tTA2-W...,759731,2024-12-12_18-53-31,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment
...,...,...,...,...,...,...,...,...
92,1365173010,snap25_oi4_dox,Snap25-IRES2-Cre/wt;Oi4(TIT2L-jGCaMP8s-RiboL1-...,726433,2024-05-09_15-22-10,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment
93,1364914325,snap25_oi4_dox,Snap25-IRES2-Cre/wt;Oi4(TIT2L-jGCaMP8s-RiboL1-...,726433,2024-05-08_15-58-27,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment
94,1366149747,snap25_oi4_dox,Snap25-IRES2-Cre/wt;Oi4(TIT2L-jGCaMP8s-RiboL1-...,726433,2024-05-13_15-09-27,OPHYS_2_images_A_passive,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment
95,1366394411,snap25_oi4_dox,Snap25-IRES2-Cre/wt;Oi4(TIT2L-jGCaMP8s-RiboL1-...,726433,2024-05-14_15-13-14,OPHYS_2_images_A_passive,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment


In [24]:
gcamp_session_info.columns

Index(['ophys_session_id', 'gcamp', 'full_genotype', 'mouse_id',
       'date_of_acquisition', 'session_type', 'session_storage_directory',
       'project'],
      dtype='object')

# Create table matching with the lims curation mouse_ids

In [23]:
mouse_ids = gcamp_session_info['mouse_id'].unique()

co_df = pd.DataFrame()
for mouse_id in mouse_ids:
    outcome, df = cdu.get_mouse_session_df(mouse_id, include_pupil=False)
    assert outcome
    df['mouse_id'] = mouse_id
    co_df = pd.concat([co_df, df], ignore_index=True)

In [44]:
gcamp_session_info['session_key'] = gcamp_session_info.apply(lambda x: f"{x['mouse_id']}_{x['date_of_acquisition'].split('_')[0]}", axis=1)

In [27]:
co_df['session_key'] = co_df.apply(lambda x: f"{x['mouse_id']}_{x['raw_data_date']}", axis=1)

In [45]:
print(f'Num sessions in lims: {len(gcamp_session_info)}')
print(f'Num sessions in code ocean: {len(co_df)}')

Num sessions in lims: 97
Num sessions in code ocean: 73


In [46]:
# assign if it's in CO
gcamp_session_info['in_co'] = gcamp_session_info['session_key'].apply(lambda x: x in co_df['session_key'].values)

In [47]:
gcamp_session_info.in_co.value_counts()

in_co
True     58
False    39
Name: count, dtype: int64

In [48]:
co_df['in_lims'] = co_df['session_key'].apply(lambda x: x in gcamp_session_info['session_key'].values)

In [49]:
co_df.in_lims.value_counts()

in_lims
True     66
False     7
Name: count, dtype: int64

In [50]:
co_df.session_key.nunique()

65

In [51]:
co_df.drop_duplicates(subset=['session_key']).in_lims.value_counts()

in_lims
True     58
False     7
Name: count, dtype: int64

In [52]:
co_df.query('in_lims == False')

,raw_data_date,processed_data_date,capsule_id,commit_id,processed_data_asset_id,raw_data_asset_id,num_provenence_data_assets,num_raw_data_asset_ids,mouse_id,session_key,in_lims
7,2024-08-21,2024-11-08,24cad997-4633-4ab3-a17d-8b0c01cad787,None,c03503ae-db17-4e08-a1cf-3a111f122967,c28d1068-4487-4389-921a-26f3c7a64241,2,1,741863,741863_2024-08-21,False
8,2024-08-22,2024-11-08,24cad997-4633-4ab3-a17d-8b0c01cad787,None,454ebf22-380a-4acb-a6bb-4b9e0e7fd141,2d229eba-30be-4f56-a0da-2749909450ad,2,1,741863,741863_2024-08-22,False
9,2024-08-23,2024-11-08,24cad997-4633-4ab3-a17d-8b0c01cad787,None,6921a957-1051-4d18-b963-c06f69414fe8,58df1d06-6b22-42c9-bc2b-c2ba1eb11ad7,2,1,741863,741863_2024-08-23,False
10,2024-08-26,2024-11-08,24cad997-4633-4ab3-a17d-8b0c01cad787,None,5489dd4f-9e70-4e61-8b73-fb56e3617400,b8fc328f-6182-42f6-8c88-b2aaff363d7d,2,1,741863,741863_2024-08-26,False
11,2024-08-27,2024-11-08,24cad997-4633-4ab3-a17d-8b0c01cad787,None,adc3583a-116e-4e6f-8740-c5237c7e03a8,c39c1164-08f9-49de-b677-a58e5eb4275b,2,1,741863,741863_2024-08-27,False
12,2024-08-29,2024-11-08,24cad997-4633-4ab3-a17d-8b0c01cad787,a85e898f9c4df58163da0836164307f595697552,6172fecd-df25-4e0b-997f-5602c30dab10,10f4e041-2ada-4067-adca-94fb960a9fc9,2,1,741863,741863_2024-08-29,False
13,2024-09-11,2024-11-08,24cad997-4633-4ab3-a17d-8b0c01cad787,None,2f8205ed-14a3-4dbf-8693-83a7e9799268,8522a742-1707-4b12-9c23-b252254c3240,2,1,741863,741863_2024-09-11,False


In [54]:
len(co_df)

73

In [58]:
pd.set_option('display.max_rows', None)
co_num_sessions = co_df.groupby('session_key').size()
co_duplicate_sessions = co_num_sessions[co_num_sessions > 1]
co_duplicate_sessions

session_key
757436_2025-01-07    4
757436_2025-01-15    3
757436_2025-01-16    2
759075_2024-12-05    2
759075_2024-12-06    2
dtype: int64

In [37]:
gcamp_session_info.session_key.nunique()

95

In [65]:
gcamp_session_info.query('mouse_id == 757436')

,ophys_session_id,gcamp,full_genotype,mouse_id,date_of_acquisition,session_type,session_storage_directory,project,session_key,in_co
42,1413427810,slc32a1_oi1,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,757436,2025-01-07_19-21-32,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment,757436_2025-01-07,True
47,1414788110,slc32a1_oi1,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,757436,2025-01-16_17-12-09,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment,757436_2025-01-16,True
48,1414595356,slc32a1_oi1,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,757436,2025-01-15_19-17-13,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment,757436_2025-01-15,True
49,1414274032,slc32a1_oi1,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,757436,2025-01-13_17-35-30,STAGE_1,\\allen\programs\mindscope\production\learning...,LearningmFISHDevelopment,757436_2025-01-13,False


In [64]:
gcamp_session_info.query('in_co == False')['session_type'].value_counts()

session_type
OPHYS_2_images_A_passive    38
STAGE_1                      1
Name: count, dtype: int64

# Create csv files for uploading and triggering
- in chunks of 13 (to be done in 3 days)

In [66]:
gcamp_session_info.columns

Index(['ophys_session_id', 'gcamp', 'full_genotype', 'mouse_id',
       'date_of_acquisition', 'session_type', 'session_storage_directory',
       'project', 'session_key', 'in_co'],
      dtype='object')

In [68]:
capsule_id = '40e54443-c0cd-4403-adb7-d3e4c69b88de'
chunk_size = 13
save_dir = Path(r'\\allen\programs\mindscope\workgroups\learning\uploading_trigger_tables')
for i in range(3):
    start = i * chunk_size
    end = start + chunk_size
    dir_names = gcamp_session_info.query('in_co == False').iloc[start:end][['session_storage_directory']]
    chunk_df = pd.DataFrame({'lims_directory': dir_names['session_storage_directory'].values})
    chunk_df['capsule_id'] = capsule_id
    chunk_df['mount'] = 'ophys_mount'
    chunk_df['force_cloud_sync'] = True

    save_fn = save_dir / f'2025-04-18-gcamp8_{i}.csv'
    chunk_df.to_csv(save_fn, index=False)